# CodeAlpha Machine Learning Internship
## Task 4: Disease Prediction from Medical Data

**Objective**: Predict patient disease risk using clinical parameters and evaluate multiple classification algorithms.
- **Dataset**: Multi-parameter Medical Clinical Records (Heart Disease & Diabetes parameters)
- **Algorithms**: Support Vector Machine (SVM), Random Forest, Logistic Regression, Decision Tree
- **Metrics**: Precision, Recall, F1-Score, ROC-AUC, 5-Fold Cross Validation

In [ ]:
# 1. Import Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, roc_curve

print("All libraries imported successfully!")

### 2. Loading Medical Clinical Dataset
Structured medical records containing clinical features: Age, Blood Pressure, Glucose, Cholesterol, BMI, Insulin, Heart Rate.

In [ ]:
# Load medical dataset from CSV file
dataset_path = "medical_disease_data.csv"
df = pd.read_csv(dataset_path)

print(f"Loaded dataset with {df.shape[0]} patient records and {df.shape[1]} features.")
df.head()

### 3. Exploratory Data Analysis & Summary Statistics

In [ ]:
print(df.info())
print("
--- Disease Distribution ---")
print(df["DiseaseTarget"].value_counts())
df.describe()

### 4. Data Preprocessing & Train-Test Split

In [ ]:
X = df.drop(columns=["DiseaseTarget"])
y = df["DiseaseTarget"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape} | Test set shape: {X_test_scaled.shape}")

### 5. Multi-Model Training and Benchmark Evaluation

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    "Support Vector Machine (SVM)": SVC(kernel="rbf", probability=True, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="accuracy")
    
    trained_models[name] = model
    
    results.append({
        "Algorithm": name,
        "Accuracy (%)": np.round(acc * 100, 2),
        "Precision": np.round(prec, 4),
        "Recall": np.round(rec, 4),
        "F1-Score": np.round(f1, 4),
        "ROC-AUC": np.round(roc_auc, 4),
        "CV Mean Accuracy (%)": np.round(cv_scores.mean() * 100, 2)
    })

results_df = pd.DataFrame(results)
results_df

### 6. Interactive Patient Risk Diagnosis Demo

In [ ]:
best_model = trained_models["Support Vector Machine (SVM)"]

def diagnose_patient(age, bp, glucose, chol, bmi, insulin, hr):
    patient_data = pd.DataFrame([[age, bp, glucose, chol, bmi, insulin, hr]], columns=X.columns)
    patient_scaled = scaler.transform(patient_data)
    pred = best_model.predict(patient_scaled)[0]
    prob = best_model.predict_proba(patient_scaled)[0][1]
    
    status = "🔴 Positive Risk (Medical Attention Advised)" if pred == 1 else "🟢 Low Risk / Healthy Profile"
    confidence = prob if pred == 1 else (1 - prob)
    
    print(f"Diagnosis: {status}")
    print(f"Model Confidence: {confidence * 100:.2f}%")

# Test with Sample Patient Profile
print("--- Sample Clinical Test ---")
diagnose_patient(age=55, bp=145, glucose=160, chol=240, bmi=31.2, insulin=140, hr=82)